# Importing the necessary libraries

In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import joblib
import os

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")


In [2]:
# Load the dataset
df = pd.read_csv('D:/ShaunakKathavate Github/digital_marketing_campaign_predictor/data/raw/raw_data.csv')

print("Shape:", df.shape)
df.head()


Shape: (8000, 20)


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


In [3]:
# Cleaning the data
# Drop duplicates
df = df.drop_duplicates()

# Drop CustomerID
df.drop("CustomerID", axis=1, inplace=True)

# Fix data types
df['Age'] = df['Age'].astype(int)
df['Income'] = df['Income'].astype(float)
df['AdSpend'] = df['AdSpend'].astype(float)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  8000 non-null   int32  
 1   Gender               8000 non-null   object 
 2   Income               8000 non-null   float64
 3   CampaignChannel      8000 non-null   object 
 4   CampaignType         8000 non-null   object 
 5   AdSpend              8000 non-null   float64
 6   ClickThroughRate     8000 non-null   float64
 7   ConversionRate       8000 non-null   float64
 8   WebsiteVisits        8000 non-null   int64  
 9   PagesPerVisit        8000 non-null   float64
 10  TimeOnSite           8000 non-null   float64
 11  SocialShares         8000 non-null   int64  
 12  EmailOpens           8000 non-null   int64  
 13  EmailClicks          8000 non-null   int64  
 14  PreviousPurchases    8000 non-null   int64  
 15  LoyaltyPoints        8000 non-null   i

In [4]:
# Handling missing values
# Numerical
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    if col in ['Income', 'AdSpend']:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mean(), inplace=True)

# Categorical
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.isnull().sum()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_26652\3492908061.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_26652\3492908061.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

Age                    0
Gender                 0
Income                 0
CampaignChannel        0
CampaignType           0
AdSpend                0
ClickThroughRate       0
ConversionRate         0
WebsiteVisits          0
PagesPerVisit          0
TimeOnSite             0
SocialShares           0
EmailOpens             0
EmailClicks            0
PreviousPurchases      0
LoyaltyPoints          0
AdvertisingPlatform    0
AdvertisingTool        0
Conversion             0
dtype: int64

In [5]:
# Outlier detection and treatment
for col in num_cols:
    if col != 'Conversion':
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)

        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])


In [6]:
# Feature engineering
# Engagement Score
df['EngagementScore'] = (
    df['WebsiteVisits'] * 0.2 +
    df['PagesPerVisit'] * 0.2 +
    df['TimeOnSite'] * 0.2 +
    df['EmailClicks'] * 0.2 +
    df['SocialShares'] * 0.2
)

# Value per Visit
df['ValuePerVisit'] = df['AdSpend'] / (df['WebsiteVisits'] + 1)

# Customer Value
df['CustomerValue'] = df['PreviousPurchases'] * df['LoyaltyPoints']


# New Features

# Spend efficiency
df['Spend_per_Click'] = df['AdSpend'] / (df['ClickThroughRate'] + 1e-5)

# Click behavior
df['Clicks_per_Visit'] = df['EmailClicks'] / (df['WebsiteVisits'] + 1)

# Deep engagement signal
df['DeepEngagement'] = df['PagesPerVisit'] * df['TimeOnSite']

# Email engagement (kept, replaces EmailCTR)
df['EmailEngagementRate'] = df['EmailClicks'] / (df['EmailOpens'] + 1)

# Customer maturity
df['CustomerMaturity'] = df['PreviousPurchases'] / (df['Age'] + 1)

df.head()

,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion,EngagementScore,ValuePerVisit,CustomerValue,Spend_per_Click,Clicks_per_Visit,DeepEngagement,EmailEngagementRate,CustomerMaturity
0,56.0,Female,136912.0,Social Media,Awareness,6497.870068,0.043919,0.088031,0.0,2.399017,7.396803,19.0,6.0,9.0,4.0,688.0,IsConfid,ToolConfid,1,7.559164,6497.870068,2752.0,147919.196897,9.000000,17.745052,1.285714,0.070175
1,69.0,Male,41760.0,Email,Retention,3898.668606,0.155725,0.182725,42.0,2.917138,5.352549,5.0,2.0,7.0,2.0,3459.0,IsConfid,ToolConfid,1,12.453937,90.666712,6918.0,25033.979700,0.162791,15.614122,2.333333,0.028571
2,46.0,Female,88456.0,PPC,Awareness,1546.429596,0.277490,0.076423,2.0,8.223619,13.794901,0.0,11.0,2.0,8.0,2337.0,IsConfid,ToolConfid,1,5.203704,515.476532,18696.0,5572.711855,0.666667,113.444015,0.166667,0.170213
3,32.0,Female,44085.0,PPC,Conversion,539.525936,0.137611,0.088004,47.0,4.540939,14.688363,89.0,2.0,2.0,0.0,2463.0,IsConfid,ToolConfid,1,31.445860,11.240124,0.0,3920.367822,0.041667,66.698958,0.666667,0.000000
4,60.0,Female,83964.0,PPC,Conversion,1678.043573,0.252851,0.109940,0.0,2.046847,13.993370,6.0,6.0,6.0,8.0,4345.0,IsConfid,ToolConfid,1,5.608044,1678.043573,34760.0,6636.226476,6.000000,28.642290,0.857143,0.131148


In [7]:
# EDA
df.describe()

,Age,Income,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,Conversion,EngagementScore,ValuePerVisit,CustomerValue,Spend_per_Click,Clicks_per_Visit,DeepEngagement,EmailEngagementRate,CustomerMaturity
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.00000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,43.625500,84662.506900,5000.951627,0.154828,0.104389,24.751625,5.549314,7.727698,49.79050,9.476875,4.467375,4.485500,2490.292775,0.876500,18.457302,430.377984,11219.258736,58931.072185,0.384437,43.064145,0.792934,0.114852
std,14.902785,37557.708335,2836.411855,0.083964,0.054844,14.312269,2.605869,4.226110,28.88557,5.711111,2.856564,2.888093,1428.635482,0.329031,6.546781,900.303751,10518.176922,85971.640483,0.822311,32.906098,1.223746,0.091178
min,18.000000,21237.980000,208.856543,0.012499,0.011872,0.000000,1.092464,0.626235,0.00000,0.000000,0.000000,0.000000,52.990000,0.000000,1.024299,4.262378,0.000000,702.287171,0.000000,0.702289,0.000000,0.000000
25%,31.000000,51744.500000,2523.221165,0.082635,0.056410,13.000000,3.302479,4.068340,25.00000,5.000000,2.000000,2.000000,1254.750000,1.000000,13.491323,99.486560,2666.250000,16210.223987,0.076923,16.175104,0.200000,0.044776
50%,43.000000,84926.500000,5013.440044,0.154505,0.104046,25.000000,5.534257,7.682956,50.00000,9.000000,4.000000,4.000000,2497.000000,1.000000,18.574351,194.866006,8153.000000,32190.817576,0.173913,34.789310,0.428571,0.100000
75%,56.000000,116815.750000,7407.989369,0.228207,0.152077,37.000000,7.835756,11.481468,75.00000,14.000000,7.000000,7.000000,3702.250000,1.000000,23.436378,370.408007,17601.250000,60912.930391,0.333333,63.552466,0.818182,0.160714
max,69.000000,148424.210000,9901.971843,0.297385,0.197881,49.000000,9.907290,14.864002,98.00000,19.000000,9.000000,9.000000,4947.000000,1.000000,34.723089,9853.176377,44523.000000,789162.221888,9.000000,147.065120,9.000000,0.473684


In [8]:
# Split features and target
X = df.drop("Conversion", axis=1)
y = df["Conversion"]

In [9]:
# Preprocessing pipelines (scaling + encoding)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

# Combine
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [10]:
# Fit and transform
X_processed = preprocessor.fit_transform(X)

In [11]:
# DataFrame with processed features
# Get encoded feature names
ohe_cols = preprocessor.named_transformers_['cat'] \
    .named_steps['encoder'] \
    .get_feature_names_out()

final_cols = list(num_cols) + list(ohe_cols)

X_processed_df = pd.DataFrame(X_processed, columns=final_cols)

# Add target
final_df = pd.concat([X_processed_df, y.reset_index(drop=True)], axis=1)

final_df.head()

,Age,Income,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,EngagementScore,ValuePerVisit,CustomerValue,Spend_per_Click,Clicks_per_Visit,DeepEngagement,EmailEngagementRate,CustomerMaturity,x0_Female,x0_Male,x1_Email,x1_PPC,x1_Referral,x1_SEO,x1_Social Media,x2_Awareness,x2_Consideration,x2_Conversion,x2_Retention,x3_IsConfid,x4_ToolConfid,Conversion
0,0.830400,1.391266,0.527784,-1.320998,-0.298267,-1.729507,-1.209000,-0.078303,-1.066014,-0.608829,1.586840,-0.168115,-1.261627,-1.664760,6.739805,-0.805062,1.035152,10.477917,-0.769483,0.402707,-0.490024,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1
1,1.702775,-1.142380,-0.388643,0.010679,1.428427,1.205221,-1.010159,-0.562053,-1.550715,-1.309262,0.886654,-0.860656,0.678107,-0.917052,-0.377353,-0.408961,-0.394307,-0.269557,-0.834245,1.258836,-0.946345,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1
2,0.159343,0.101011,-1.217996,1.460971,-0.509946,-1.589758,1.026327,1.435737,-1.723823,0.266712,-0.863810,1.216969,-0.107307,-2.024572,0.094528,0.710884,-0.620690,0.343237,2.138943,-0.511795,0.607205,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1
3,-0.780138,-1.080472,-1.573010,-0.205066,-0.298763,1.554594,-0.386987,1.647165,1.357493,-1.309262,-0.863810,-1.553198,-0.019105,1.984085,-0.465581,-1.066721,-0.639910,-0.416864,0.718295,-0.103188,-1.259722,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1
4,1.098823,-0.018599,-1.171591,1.167504,0.101230,-1.729507,-1.344153,1.482702,-1.516094,-0.608829,0.536561,1.216969,1.298318,-1.962806,1.385914,2.238241,-0.608318,6.829433,-0.438300,0.052472,0.178730,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1


In [12]:
# Save processed data
os.makedirs("D:/ShaunakKathavate Github/digital_marketing_campaign_predictor/data/processed", exist_ok=True)
final_df.to_csv("D:/ShaunakKathavate Github/digital_marketing_campaign_predictor/data/processed/processed_data.csv", index=False)

# Save pipeline
os.makedirs("D:/ShaunakKathavate Github/digital_marketing_campaign_predictor/models", exist_ok=True)
joblib.dump(preprocessor, "D:/ShaunakKathavate Github/digital_marketing_campaign_predictor/models/data_preprocessor.pkl")

print("✅ Data and pipeline saved successfully!")


✅ Data and pipeline saved successfully!
